<a href="https://colab.research.google.com/github/Mohamedalaa822/Image-processing-project-/blob/main/nlp_emotions_classification_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Text_emotion_classification

#1-Load Data

In [ ]:
!pip install datasets

In [ ]:
from datasets import load_dataset
dataset = load_dataset("dair-ai/emotion")

In [ ]:
print(dataset)

In [ ]:
print(dataset["train"][0])

In [ ]:
import pandas as pd
df = dataset["train"].to_pandas()
df.to_csv("emotion_train.csv", index=False)

#2-Load Libraries

In [ ]:
import numpy as np
import pandas as pd
import pickle
import nltk
import re
import string
from sklearn.preprocessing import LabelEncoder
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import seaborn as sns
import matplotlib.pyplot as plt
from wordcloud import WordCloud

In [ ]:
# Download required NLTK data
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')

In [ ]:
import pandas as pd
df = pd.read_csv(
    "emotion_train.csv",
    sep=","
)
df["length"] = df["text"].apply(len)

In [ ]:
# Label mapping
LABEL_NAMES = dataset["train"].features["label"].names
NUM_LABELS  = len(LABEL_NAMES)
id2label    = {i: l for i, l in enumerate(LABEL_NAMES)}
label2id    = {l: i for i, l in enumerate(LABEL_NAMES)}
print("Labels:", LABEL_NAMES)

In [ ]:
# Preview a few examples
df_train = dataset["train"].to_pandas()
df_train["emotion"] = df_train["label"].map(id2label)
df_train.head(10)

In [ ]:
df_train.head()

In [ ]:
df_train.tail()

In [ ]:
df_train.info()

In [ ]:
df_train.shape

In [ ]:
df_train.isnull().sum()

In [ ]:
df_train.duplicated().sum()


In [ ]:
df_train.drop_duplicates(inplace=True)

In [ ]:
df_train.duplicated().sum()

In [ ]:
df_train['label'].value_counts()

In [ ]:
df_train["length"] = df_train["text"].apply(len)
df_train["length"].describe()

#3-Exploratory Data Analysis (ُEDA)

In [ ]:
# General label mapping
plt.figure(figsize=(8, 5))
sns.countplot(
    x='emotion',
    data=df_train,
    order=df_train['emotion'].value_counts().index
)
plt.title('Emotion Distribution')
plt.xlabel('Emotion')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Add 'length' column to df_train
df_train['length'] = df_train['text'].apply(len)
# Plot histogram
sns.histplot(
    data=df_train,
    x='length',
    hue='emotion',
    multiple='stack'
)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['length'], bins=30, kde=True, color='skyblue')
plt.title('Distribution of Comment Lengths')
plt.xlabel('Number of Characters')
plt.ylabel('Frequency')
plt.show()

Text Length Analysis

In [ ]:
#Average Length
print('Average Length : ', df_train['length'].mean())

In [ ]:
#maximum Lenght
print('Maximum Length : ', df_train['length'].max())

In [ ]:
# Minimum Lenght
print('Minimum Length : ', df_train['length'].min())


In [ ]:
#Text Length Per Emotion
plt.figure(figsize=(10,6))
sns.boxplot(
    x='emotion',
    y='length',
    data=df_train
)

plt.title('Text Length by Emotion')
plt.xlabel('Emotion')
plt.ylabel('Length')

plt.xticks(rotation=45)

plt.show()

In [ ]:
# Words cloud for each emotions
def words_cloud(wordcloud_obj, emotion_name):
    plt.figure(figsize=(10, 10))
    plt.title(emotion_name + ' Word Cloud', size = 16)
    plt.imshow(wordcloud_obj)
    plt.axis("off");

emotions_list = df_train['emotion'].unique()
for emotion in emotions_list:
    text = ' '.join([sentence for sentence in df_train.loc[df_train['emotion'] == emotion,'text']])
    wordcloud = WordCloud(width = 600, height = 600).generate(text)
    words_cloud(wordcloud, emotion)

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, split in zip(axes, ["train", "validation", "test"]):
    df = dataset[split].to_pandas()
    counts = df["label"].map(id2label).value_counts()
    sns.barplot(x=counts.index, y=counts.values, palette="Set2", ax=ax)
    ax.set_title(f"{split} split ({len(df)} samples)")
    ax.set_xlabel("Emotion")
    ax.set_ylabel("Count")
    for bar in ax.patches:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 20,
                int(bar.get_height()),
                ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.suptitle("Class Distribution per Split", y=1.02, fontsize=14, fontweight="bold")
plt.show()

In [ ]:
#sampeltext from each emotion
for emotion in df_train['emotion'].unique():
    sample = df_train[df_train['emotion'] == emotion]['text'].iloc[0]
    print(f'Emotion : {emotion}')
    print(sample)
    print('-' * 50)

#4-Data Preprocessing

In [ ]:
df_train

In [ ]:
# Initialize lemmatizer and stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Cleaning function
def clean_text(text):
    text = re.sub(r"[^a-zA-Z]", " ", text)
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    words = word_tokenize(text)
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return " ".join(words)

# Apply to DataFrame
df_train['cleaned_text'] = df_train['text'].astype(str).apply(clean_text)

In [ ]:
df_train

#5-Applying machine learning classifier

In [ ]:
# ML packages
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [ ]:
df2=df_train.copy()
df2

In [ ]:
X = df_train['cleaned_text']
y = df_train['label']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 3),
    stop_words='english',
    lowercase=True,
    strip_accents='unicode',
    sublinear_tf=True
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

In [ ]:
#Define the 6 classifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import SGDClassifier

SEED = 42
classifiers = {
    "Multinomial Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(),
    "Support Vector Machine": SVC(),
    "SGD Classifier (SVM alternative)": SGDClassifier(loss="hinge", max_iter=3000, tol=1e-3),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=7, metric="cosine"),
}

print("Classifiers defined:", list(classifiers.keys()))

In [ ]:
#Train & evaluate all 6 classifiers
import time
from sklearn.metrics import accuracy_score, f1_score, classification_report
ml_results = {}
for name, clf in  classifiers.items():
    t0 = time.time()
    clf.fit(X_train_tfidf, y_train)
    train_time = time.time() - t0
    y_pred_test = clf.predict(X_test_tfidf)
    test_acc = accuracy_score(y_test, y_pred_test)
    test_f1  = f1_score(y_test, y_pred_test, average="weighted")

    ml_results[name] = {
        "test_acc":  test_acc,
        "test_f1":   test_f1,
        "train_sec": train_time,
        "clf":       clf,
        "y_pred_test": y_pred_test,
    }

    print(f"[{name:30s}]  Test Acc: {test_acc:.4f}  Test F1: {test_f1:.4f}  ({train_time:.1f}s)")

Models evaluations

In [ ]:
# Leaderboard table
import pandas as pd

rows = []
for name, r in ml_results.items():
    rows.append({
        "Model":          name,
        "Test Acc":       f"{r['test_acc']:.4f}",
        "Test F1 (W)":    f"{r['test_f1']:.4f}",
        "Train Time (s)": f"{r['train_sec']:.1f}",
        "Test_Preds":     r["y_pred_test"]
    })

df_ml = pd.DataFrame(rows).sort_values("Test F1 (W)", ascending=False).reset_index(drop=True)
df_ml.index += 1
print(df_ml.to_string())

In [ ]:
# Bar-chart comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

names  = list(ml_results.keys())
accs   = [ml_results[n]["test_acc"] for n in names]
f1s    = [ml_results[n]["test_f1"]  for n in names]

palette = sns.color_palette("tab10", len(names))

for ax, values, title, ylabel in [
    (axes[0], accs, "Test Accuracy — ML Classifiers",       "Accuracy"),
    (axes[1], f1s,  "Test F1 Weighted — ML Classifiers",    "F1 (Weighted)"),
]:
    bars = ax.bar(names, values, color=palette, edgecolor="black", linewidth=0.6)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight="bold")
    ax.set_xticklabels(names, rotation=30, ha="right")
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.show()


In [ ]:
# Per-class report for best ML model
best_ml_name = max(ml_results, key=lambda n: ml_results[n]["test_f1"])
best_ml_pred = ml_results[best_ml_name]["y_pred_test"]

print(f"Best ML model: {best_ml_name}")
print()
print("=== Classification Report ===")
print(classification_report(y_test, best_ml_pred, target_names=LABEL_NAMES, digits=4))


In [ ]:
# Confusion matrix for best ML model
cm_ml      = confusion_matrix(y_test, best_ml_pred)
cm_ml_norm = cm_ml.astype(float) / cm_ml.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, data, fmt, title in zip(
    axes,
    [cm_ml, cm_ml_norm],
    ["d", ".2f"],
    [f"Confusion Matrix — {best_ml_name} (counts)",
     f"Confusion Matrix — {best_ml_name} (normalised)"],
):
    sns.heatmap(data, annot=True, fmt=fmt, cmap="Oranges",
                xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
                ax=ax, linewidths=0.5)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
plt.tight_layout()
plt.show()

Predictive System (Machine Learning classfire)

In [ ]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier

# Prediction Function
def predict_emotion(input_text, model):
    cleaned_text = clean_text(input_text)
    input_vectorized = tfidf_vectorizer.transform([cleaned_text])
    predicted_label = model.predict(input_vectorized)[0]
    predicted_emotion = id2label[predicted_label]
    return predicted_emotion, predicted_label


# Models
models = {

    "Logistic Regression":
        LogisticRegression(
            max_iter=2000,
            C=5.0,
            solver="lbfgs",
            random_state=SEED
        ),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            n_jobs=-1,
            random_state=SEED
        ),
    "SVM":
        SGDClassifier(
            loss="hinge",
            max_iter=3000,
            tol=1e-3,
            random_state=SEED
        ),
    "Gradient Boosting":
        GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.1,
            max_depth=5,
            random_state=SEED
        ),
    "Multinomial Naive Bayes":
        MultinomialNB(
            alpha=0.1
        ),
    "K-Nearest Neighbors":
        KNeighborsClassifier(
            n_neighbors=7,
            metric="cosine",
            n_jobs=-1
        ),
}
# Train Models
for name, clf in models.items():

    print(f"Training {name}...")

    clf.fit(X_train_tfidf, y_train)

print("\nAll models trained successfully!\n")


# Test Sentences
sentences = [
    "i didnt feel humiliated",
    "i feel strong and good overall",
    "im grabbing a minute to post i feel greedy wrong",
    "He was speechless when he found out he was accepted to this new job",
    "This is outrageous, how can you talk like that?",
    "I feel like im all alone in this world",
    "He is really sweet and caring",
    "You made me very crazy",
    "i am ever feeling nostalgic about the fireplace i will know that it is still on the property",
    "i am feeling grouchy",
    "He hates you"
]


# Select Model
selected_model = models["SVM"]
# Predictions
for sentence in sentences:
    pred_emotion, label = predict_emotion(sentence, selected_model)
    print(f"Text       : {sentence}")
    print(f"Prediction : {pred_emotion}")
    print(f"Label      : {label}")
    print("=" * 60)

In [ ]:
import pickle


# Save Your Models


pickle.dump(
    models["Logistic Regression"],
    open("logistic_regression.pkl", "wb")
)

pickle.dump(
    models["Random Forest"],
    open("random_forest.pkl", "wb")
)

pickle.dump(
    models["SVM"],
    open("svm_model.pkl", "wb")
)

pickle.dump(
    models["Gradient Boosting"],
    open("gradient_boosting.pkl", "wb")
)

pickle.dump(
    models["Multinomial Naive Bayes"],
    open("multinomial_nb.pkl", "wb")
)

pickle.dump(
    models["K-Nearest Neighbors"],
    open("knn_model.pkl", "wb")
)

# Save TF-IDF Vectorizer

pickle.dump(
    tfidf_vectorizer,
    open("tfidf_vectorizer.pkl", "wb")
)

# Save Label Encoder (or id2label mapping)

pickle.dump(
    id2label,
    open("id2label.pkl", "wb") # Changed from lb to id2label
)
print("All models saved successfully!")

# 6-Applying DeepLearning (LSTM)

In [ ]:
!pip install -q transformers datasets evaluate accelerate seaborn scikit-learn

In [ ]:
# DL packages
import numpy as np
import re
import string
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, LSTM, Bidirectional, Dense, Dropout,
    SpatialDropout1D, BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.regularizers import l2
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)

print("All Deep LSTM packages imported successfully!")

In [ ]:
# Text cleaning function
def text_cleaning(df2, column, vocab_size, max_len):
    corpus = []

    for text in df2[column]:
        text = re.sub(r"[^a-zA-Z]", " ", str(text))
        text = text.lower()
        words = word_tokenize(text)
        words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
        cleaned_text = " ".join(words)
        corpus.append(cleaned_text)

    # One-hot encode and pad
    one_hot_word = [one_hot(input_text=text, n=vocab_size) for text in corpus]
    pad = pad_sequences(sequences=one_hot_word, maxlen=max_len, padding='pre')
    return pad

Model Buliding and training

In [ ]:
# Hyperparameters
VOCAB_SIZE  = 20000
MAX_LEN     = 100
EMBED_DIM   = 128
LSTM_UNITS  = 128
DROPOUT     = 0.3
BATCH_SIZE  = 64
EPOCHS      = 20
SEED        = 42

np.random.seed(SEED)
print("Hyperparameters set.")

In [ ]:
# Prepare train / val / test splits
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt',     quiet=True)
nltk.download('wordnet',   quiet=True)
nltk.download('punkt_tab', quiet=True)
lemmatizer = WordNetLemmatizer()
stop_words  = set(stopwords.words('english'))
def clean_for_lstm(text):
    text  = re.sub(r"[^a-zA-Z]", " ", str(text))
    text  = text.lower()
    words = word_tokenize(text)
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    return " ".join(words)

# Apply cleaning to each split
train_texts  = [clean_for_lstm(t) for t in dataset["train"]["text"]]
val_texts    = [clean_for_lstm(t) for t in dataset["validation"]["text"]]
test_texts   = [clean_for_lstm(t) for t in dataset["test"]["text"]]

train_labels = dataset["train"]["label"]
val_labels   = dataset["validation"]["label"]
test_labels  = dataset["test"]["label"]

print(f"Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}")


In [ ]:
# Tokenise & Pad
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(train_texts)

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(train_texts),
                             maxlen=MAX_LEN, padding='pre', truncating='post')
X_val_seq   = pad_sequences(tokenizer.texts_to_sequences(val_texts),
                             maxlen=MAX_LEN, padding='pre', truncating='post')
X_test_seq  = pad_sequences(tokenizer.texts_to_sequences(test_texts),
                             maxlen=MAX_LEN, padding='pre', truncating='post')

NUM_CLASSES = len(LABEL_NAMES)
y_train_cat = to_categorical(train_labels, num_classes=NUM_CLASSES)
y_val_cat   = to_categorical(val_labels,   num_classes=NUM_CLASSES)
y_test_cat  = to_categorical(test_labels,  num_classes=NUM_CLASSES)

print(f"X_train shape : {X_train_seq.shape}")
print(f"X_val   shape : {X_val_seq.shape}")
print(f"X_test  shape : {X_test_seq.shape}")
print(f"Classes       : {NUM_CLASSES} → {LABEL_NAMES}")


In [ ]:
# Build and compile the model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    LSTM,
    Dense,
    Dropout,
    BatchNormalization,
    SpatialDropout1D
)
from tensorflow.keras.callbacks import EarlyStopping
# Build Model
model = Sequential([

    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBED_DIM,
        input_length=MAX_LEN
    ),

    SpatialDropout1D(0.3),
    # Single LSTM Layer
    LSTM(
        LSTM_UNITS,
        dropout=0.3,
        recurrent_dropout=0.2
    ),

    BatchNormalization(),

    Dense(128, activation='relu'),

    Dropout(0.5),

    Dense(NUM_CLASSES, activation='softmax')

])
# Compile
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
# Early Stopping
callback = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)
# Train
history = model.fit(
    X_train_seq,
    y_train_cat,
    epochs=10,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_seq, y_val_cat),
    callbacks=[callback],
    verbose=1
)

In [ ]:
# Training Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'],     label='Train Acc',  linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val Acc',    linewidth=2, linestyle='--')
axes[0].set_title('Deep LSTM — Accuracy',    fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history['loss'],     label='Train Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Val Loss',   linewidth=2, linestyle='--')
axes[1].set_title('Deep LSTM — Loss', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Deep Bidirectional LSTM — Training History',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Evaluation on Test Set
loss_test, acc_test = model.evaluate(X_test_seq, y_test_cat, verbose=0)
y_pred_prob  = model.predict(X_test_seq, verbose=0)
y_pred_class = np.argmax(y_pred_prob, axis=1)
y_true_class = np.array(test_labels)
f1_weighted  = f1_score(y_true_class, y_pred_class, average='weighted')
f1_macro     = f1_score(y_true_class, y_pred_class, average='macro')
print("=" * 55)
print("         Deep Bidirectional LSTM — Test Results")
print("=" * 55)
print(f"  Test Loss        : {loss_test:.4f}")
print(f"  Test Accuracy    : {acc_test:.4f}")
print(f"  F1 Weighted      : {f1_weighted:.4f}")
print(f"  F1 Macro         : {f1_macro:.4f}")
print("=" * 55)
print()
print("=== Classification Report ===")
print(classification_report(y_true_class, y_pred_class,
                            target_names=LABEL_NAMES, digits=4))

In [ ]:
# Predictive System — Deep LSTM
EMOJI_MAP = {
    'sadness': '😔', 'joy': '😄', 'love': '❤️',
    'anger': '😡', 'fear': '😨', 'surprise': '😲'
}
def predict_emotion_lstm(input_text, model=model,
                          tok=tokenizer, max_len=MAX_LEN):
    """
    Predict the emotion of a raw text string using the Deep LSTM model.

    Returns:
        emotion  (str)   – predicted emotion label
        label_id (int)   – numeric label
        confidence (float) – model confidence (max softmax probability)
        probs    (dict)  – probability for every class
    """
    cleaned  = clean_for_lstm(input_text)
    seq      = tokenizer.texts_to_sequences([cleaned])
    padded   = pad_sequences(seq, maxlen=max_len,
                             padding='pre', truncating='post')

    prob_vec    = model.predict(padded, verbose=0)[0]
    label_id    = int(np.argmax(prob_vec))
    emotion     = id2label[label_id]
    confidence  = float(prob_vec[label_id])
    all_probs   = {id2label[i]: float(prob_vec[i]) for i in range(len(prob_vec))}

    return emotion, label_id, confidence, all_probs


# Test on sample sentences
test_sentences = [
    "i didnt feel humiliated",
    "i feel strong and good overall",
    "im grabbing a minute to post i feel greedy wrong",
    "He was speechless when he found out he was accepted to this new job",
    "This is outrageous, how can you talk like that?",
    "I feel like im all alone in this world",
    "He is really sweet and caring",
    "You made me very crazy",
    "i am ever feeling nostalgic about the fireplace",
    "i am feeling grouchy",
    "He hates you",
]

print("=" * 65)
print("          Deep BiLSTM — Emotion Prediction Results")
print("=" * 65)
for sentence in test_sentences:
    emotion, lid, conf, probs = predict_emotion_lstm(sentence)
    emoji = EMOJI_MAP.get(emotion, '')
    print(f"Text       : {sentence}")
    print(f"Prediction : {emotion.upper()} {emoji}  (confidence: {conf:.1%})")
    # Show top-3 probabilities
    top3 = sorted(probs.items(), key=lambda x: x[1], reverse=True)[:3]
    print(f"Top-3      : " + "  |  ".join(f"{e}: {p:.1%}" for e, p in top3))
    print("-" * 65)


In [ ]:
# Save Deep LSTM Model
import pickle, os
# Save Keras model (full model with weights)
model.save("deep_bilstm_emotion.keras")
# Save the tokenizer so we can reload it for inference
with open("lstm_tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)
# Save id2label mapping (if not already saved)
with open("id2label.pkl", "wb") as f:
    pickle.dump(id2label, f)

print(" Model saved  →  deep_lstm_emotion.keras")
print("Tokenizer saved  →  lstm_tokenizer.pkl")
print(" id2label saved   →  id2label.pkl")